In [1]:
# K-Means Clustering with NLP — `us_cfr.csv`

## 1. Install / Import Libraries

```python
# If needed, run this once in a Jupyter cell:
# !pip install pandas numpy scikit-learn nltk matplotlib seaborn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Download NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
```

---

## 2. Load the Dataset

Make sure `us_cfr.csv` is in the same folder as your Jupyter Notebook.

```python
df = pd.read_csv("us_cfr.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

display(df.head())
```

---

## 3. Check the Dataset

```python
print(df.info())

print("\nMissing values:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:", df.duplicated().sum())
```

Remove duplicate rows:

```python
df = df.drop_duplicates().reset_index(drop=True)

print("New dataset shape:", df.shape)
```

---

## 4. Find the Text Column

First, inspect the columns and their data types:

```python
for column in df.columns:
    print(f"{column}: {df[column].dtype}")
```

You can then select the column containing the CFR text.

For example, if your text is stored in a column called `text`:

```python
TEXT_COLUMN = "text"
```

If your dataset uses a different name, change it accordingly.

You can also automatically select the object/string column with the most text:

```python
if TEXT_COLUMN not in df.columns:
    text_columns = df.select_dtypes(include=["object"]).columns

    if len(text_columns) == 0:
        raise ValueError("No text column was found in the dataset.")

    avg_lengths = {
        col: df[col].fillna("").astype(str).str.len().mean()
        for col in text_columns
    }

    TEXT_COLUMN = max(avg_lengths, key=avg_lengths.get)

print("Selected text column:", TEXT_COLUMN)
```

---

## 5. Clean the Text Using NLP

Create an NLP preprocessing function:

```python
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()

    # Remove URLs
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text)

    # Remove HTML
    text = re.sub(r"<.*?>", " ", text)

    # Keep alphabetic characters
    text = re.sub(r"[^a-z\s]", " ", text)

    # Tokenize
    words = text.split()

    # Remove stopwords and very short words
    words = [
        word for word in words
        if word not in stop_words and len(word) > 2
    ]

    # Lemmatization
    words = [lemmatizer.lemmatize(word) for word in words]

    return " ".join(words)
```

Apply the cleaning:

```python
df["clean_text"] = df[TEXT_COLUMN].fillna("").apply(clean_text)

display(df[[TEXT_COLUMN, "clean_text"]].head())
```

---

## 6. Remove Empty Documents

Some rows may contain no usable text after preprocessing.

```python
df = df[df["clean_text"].str.strip() != ""].reset_index(drop=True)

print("Dataset shape after cleaning:", df.shape)
```

---

## 7. Convert Text into TF-IDF Features

TF-IDF converts the text into numerical features that can be used by K-Means.

```python
vectorizer = TfidfVectorizer(
    max_features=5000,
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df["clean_text"])

print("TF-IDF matrix shape:", X.shape)
```

Here:

* `max_features=5000` limits the vocabulary.
* `ngram_range=(1, 2)` uses both individual words and two-word phrases.
* TF-IDF gives more importance to words that distinguish documents.

---

## 8. Find a Suitable Number of Clusters

K-Means requires us to choose the number of clusters (`K`).

The **Elbow Method** can help.

```python
inertias = []

K_range = range(2, 11)

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    kmeans.fit(X)

    inertias.append(kmeans.inertia_)
```

Plot the results:

```python
plt.figure(figsize=(8, 5))

plt.plot(K_range, inertias, marker="o")

plt.xlabel("Number of clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.show()
```

Look for the point where the decrease in inertia begins to slow down.

---

## 9. Silhouette Score

We can also compare different values of K using the silhouette score.

```python
silhouette_scores = []

for k in K_range:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X)

    score = silhouette_score(X, labels)

    silhouette_scores.append(score)

    print(f"K={k}: Silhouette Score={score:.4f}")
```

Plot the scores:

```python
plt.figure(figsize=(8, 5))

plt.plot(
    K_range,
    silhouette_scores,
    marker="o"
)

plt.xlabel("Number of clusters (K)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different K Values")

plt.show()
```

---

## 10. Run K-Means

Choose a value of `K` based on the elbow plot and silhouette results.

For example:

```python
K = 5
```

Run K-Means:

```python
kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

df["cluster"] = kmeans.fit_predict(X)

print(df["cluster"].value_counts().sort_index())
```

---

## 11. Display Documents in Each Cluster

```python
for cluster in sorted(df["cluster"].unique()):
    
    print("\n" + "=" * 80)
    print(f"CLUSTER {cluster}")
    print("=" * 80)

    cluster_data = df[df["cluster"] == cluster]

    for i, text in enumerate(cluster_data[TEXT_COLUMN].head(5), 1):
        print(f"\nDocument {i}:")
        print(str(text)[:1000])
```

---

## 12. Find the Most Important Words in Each Cluster

The K-Means cluster centers can be used to identify important terms.

```python
terms = vectorizer.get_feature_names_out()
order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

for cluster in range(K):
    
    print("\n" + "=" * 60)
    print(f"CLUSTER {cluster} - TOP WORDS")
    print("=" * 60)

    top_words = [
        terms[ind]
        for ind in order_centroids[cluster, :15]
    ]

    print(", ".join(top_words))
```

---

## 13. Visualise the Clusters with PCA

TF-IDF can contain thousands of dimensions, so PCA can reduce it to two dimensions for visualisation.

```python
pca = PCA(n_components=2, random_state=42)

X_pca = pca.fit_transform(X.toarray())

print("Explained variance:", pca.explained_variance_ratio_)
```

Create the plot:

```python
plt.figure(figsize=(10, 7))

sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=df["cluster"],
    palette="tab10",
    s=60,
    alpha=0.7
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("K-Means Clusters of US CFR Documents")

plt.legend(title="Cluster")

plt.show()
```

---

## 14. Number of Documents in Each Cluster

```python
cluster_counts = df["cluster"].value_counts().sort_index()

plt.figure(figsize=(8, 5))

sns.barplot(
    x=cluster_counts.index,
    y=cluster_counts.values
)

plt.xlabel("Cluster")
plt.ylabel("Number of Documents")
plt.title("Documents per K-Means Cluster")

plt.show()
```

---

## 15. Save the Results

Save the cleaned text and cluster assignment:

```python
df.to_csv(
    "us_cfr_kmeans_results.csv",
    index=False
)

print("Results saved to us_cfr_kmeans_results.csv")
```

---

## 16. Display the Final Results

```python
result_columns = [
    TEXT_COLUMN,
    "clean_text",
    "cluster"
]

display(df[result_columns].head(20))
```

---

## 17. Final Cluster Summary

```python
print("Final clustering summary")
print("-" * 40)

for cluster in sorted(df["cluster"].unique()):
    
    count = (df["cluster"] == cluster).sum()

    top_words = [
        terms[ind]
        for ind in order_centroids[cluster, :10]
    ]

    print(f"\nCluster {cluster}")
    print(f"Documents: {count}")
    print("Top terms:", ", ".join(top_words))
```

### Workflow

The notebook follows this pipeline:

```text
us_cfr.csv
     |
     v
Load Dataset
     |
     v
Select Text Column
     |
     v
NLP Cleaning
     |
     +--> lowercase
     +--> remove URLs/HTML
     +--> remove punctuation
     +--> remove stopwords
     +--> lemmatization
     |
     v
TF-IDF
     |
     v
Choose K
(Elbow + Silhouette)
     |
     v
K-Means
     |
     v
Cluster Documents
     |
     +--> Top words
     +--> Cluster sizes
     +--> PCA visualisation
     |
     v
us_cfr_kmeans_results.csv
```


SyntaxError: invalid syntax (3806722441.py, line 5)